# Transformer Architecture from Scratch (NumPy)

This notebook implements the **Transformer** architecture from the paper
["Attention Is All You Need" (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762)
using only NumPy — no PyTorch or TensorFlow required.

## Architecture Overview

```
Input → Embedding + Positional Encoding → [Encoder × N] → Encoder Output
                                                              ↓
Target → Embedding + Positional Encoding → [Decoder × N] → Linear → Softmax → Output
```

**Key components built below:**
1. Positional Encoding (sinusoidal)
2. Scaled Dot-Product Attention
3. Multi-Head Attention
4. Position-wise Feed-Forward Network
5. Encoder (with residual connections + layer norm)
6. Decoder (with masked self-attention + cross-attention)
7. Full Transformer

In [1]:
import numpy as np

np.random.seed(42)

def softmax(x, axis=-1):
    """Numerically stable softmax."""
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def layer_norm(x, eps=1e-6):
    """Layer normalization over the last dimension."""
    mean = np.mean(x, axis=-1, keepdims=True)
    std = np.std(x, axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

def relu(x):
    return np.maximum(0, x)

def init_weights(shape):
    """Xavier/Glorot uniform initialization."""
    limit = np.sqrt(6.0 / sum(shape))
    return np.random.uniform(-limit, limit, size=shape)

print("Helper functions ready.")

Helper functions ready.


## 1. Positional Encoding

Since the Transformer has no recurrence or convolution, it needs positional information injected.
Uses sinusoidal functions so the model can learn to attend by relative position:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [2]:
class PositionalEncoding:
    def __init__(self, d_model, max_len=5000):
        self.encoding = np.zeros((max_len, d_model))
        positions = np.arange(max_len)[:, np.newaxis]
        div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

        self.encoding[:, 0::2] = np.sin(positions * div_term)
        self.encoding[:, 1::2] = np.cos(positions * div_term)

    def __call__(self, seq_len):
        return self.encoding[:seq_len]

# Quick test
pe = PositionalEncoding(d_model=8, max_len=100)
print("Positional encoding shape for seq_len=5:", pe(5).shape)
print("First 3 positions:\n", np.round(pe(3), 3))

Positional encoding shape for seq_len=5: (5, 8)
First 3 positions:
 [[ 0.     1.     0.     1.     0.     1.     0.     1.   ]
 [ 0.841  0.54   0.1    0.995  0.01   1.     0.001  1.   ]
 [ 0.909 -0.416  0.199  0.98   0.02   1.     0.002  1.   ]]


## 2. Scaled Dot-Product Attention

The core of the Transformer. Computes attention weights between queries and keys, then applies them to values:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

An optional mask is used in the decoder to prevent attending to future tokens.

In [3]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: shape (..., seq_len, d_k)
    mask: boolean array where True = position to mask (set to -inf)
    Returns: attention output and attention weights
    """
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-2, -1) / np.sqrt(d_k)

    if mask is not None:
        scores = np.where(mask, -1e9, scores)

    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

# Quick test: 1 sample, 4 tokens, d_k=8
Q = np.random.randn(1, 4, 8)
K = np.random.randn(1, 4, 8)
V = np.random.randn(1, 4, 8)
out, attn = scaled_dot_product_attention(Q, K, V)
print("Attention output shape:", out.shape)
print("Attention weights (row sums should be ~1):", np.round(attn[0].sum(axis=-1), 4))

Attention output shape: (1, 4, 8)
Attention weights (row sums should be ~1): [1. 1. 1. 1.]


## 3. Multi-Head Attention

Instead of one attention function, the model projects Q, K, V into `h` heads, runs attention in parallel, and concatenates:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \cdot W^O$$

where each $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$

In [4]:
class MultiHeadAttention:
    def __init__(self, d_model, num_heads):
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = init_weights((d_model, d_model))
        self.W_k = init_weights((d_model, d_model))
        self.W_v = init_weights((d_model, d_model))
        self.W_o = init_weights((d_model, d_model))

    def split_heads(self, x):
        """(batch, seq_len, d_model) → (batch, num_heads, seq_len, d_k)"""
        batch_size, seq_len, _ = x.shape
        x = x.reshape(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(0, 2, 1, 3)

    def combine_heads(self, x):
        """(batch, num_heads, seq_len, d_k) → (batch, seq_len, d_model)"""
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(0, 2, 1, 3)
        return x.reshape(batch_size, seq_len, self.d_model)

    def __call__(self, Q, K, V, mask=None):
        Q = self.split_heads(Q @ self.W_q)
        K = self.split_heads(K @ self.W_k)
        V = self.split_heads(V @ self.W_v)

        if mask is not None and mask.ndim == 3:
            mask = mask[:, np.newaxis, :, :]  # broadcast across heads

        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        output = self.combine_heads(attn_output)
        return output @ self.W_o, attn_weights

# Quick test
mha = MultiHeadAttention(d_model=16, num_heads=4)
x = np.random.randn(2, 5, 16)  # batch=2, seq=5, d_model=16
out, weights = mha(x, x, x)
print("Multi-Head Attention output shape:", out.shape)
print("Attention weights shape:", weights.shape)

Multi-Head Attention output shape: (2, 5, 16)
Attention weights shape: (2, 4, 5, 5)


## 4. Position-wise Feed-Forward Network

Applied to each position independently:

$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

Typically expands to 4× the model dimension, then projects back.

In [5]:
class FeedForward:
    def __init__(self, d_model, d_ff):
        self.W1 = init_weights((d_model, d_ff))
        self.b1 = np.zeros(d_ff)
        self.W2 = init_weights((d_ff, d_model))
        self.b2 = np.zeros(d_model)

    def __call__(self, x):
        return relu(x @ self.W1 + self.b1) @ self.W2 + self.b2

# Quick test
ff = FeedForward(d_model=16, d_ff=64)
x = np.random.randn(2, 5, 16)
print("Feed-Forward output shape:", ff(x).shape)

Feed-Forward output shape: (2, 5, 16)


## 5. Encoder

Each encoder layer has:
1. **Multi-Head Self-Attention** + residual connection + layer norm
2. **Feed-Forward Network** + residual connection + layer norm

The full encoder stacks N such layers.

In [6]:
class EncoderLayer:
    def __init__(self, d_model, num_heads, d_ff):
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)

    def __call__(self, x, mask=None):
        # Sub-layer 1: self-attention + add & norm
        attn_out, _ = self.self_attn(x, x, x, mask)
        x = layer_norm(x + attn_out)

        # Sub-layer 2: feed-forward + add & norm
        ff_out = self.ff(x)
        x = layer_norm(x + ff_out)
        return x


class Encoder:
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_len=5000):
        self.embedding = init_weights((vocab_size, d_model))
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = [EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)]

    def __call__(self, token_ids, mask=None):
        seq_len = token_ids.shape[1]
        # Token embedding + positional encoding
        x = self.embedding[token_ids]  # (batch, seq_len, d_model)
        x = x + self.pos_enc(seq_len)

        for layer in self.layers:
            x = layer(x, mask)
        return x

# Quick test
encoder = Encoder(num_layers=2, d_model=16, num_heads=4, d_ff=64, vocab_size=100)
sample_ids = np.array([[1, 5, 12, 8, 3]])  # batch=1, seq=5
enc_out = encoder(sample_ids)
print("Encoder output shape:", enc_out.shape)

Encoder output shape: (1, 5, 16)


## 6. Decoder

Each decoder layer has **three** sub-layers:
1. **Masked Multi-Head Self-Attention** — prevents attending to future positions
2. **Multi-Head Cross-Attention** — attends to encoder output (K, V from encoder; Q from decoder)
3. **Feed-Forward Network**

Each sub-layer has residual connection + layer norm.

In [7]:
def causal_mask(seq_len):
    """Upper-triangular mask: True where attention should be blocked (future tokens)."""
    return np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)


class DecoderLayer:
    def __init__(self, d_model, num_heads, d_ff):
        self.masked_self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)

    def __call__(self, x, enc_output, src_mask=None, tgt_mask=None):
        # Sub-layer 1: masked self-attention
        attn_out, _ = self.masked_self_attn(x, x, x, tgt_mask)
        x = layer_norm(x + attn_out)

        # Sub-layer 2: cross-attention (Q from decoder, K/V from encoder)
        cross_out, _ = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = layer_norm(x + cross_out)

        # Sub-layer 3: feed-forward
        ff_out = self.ff(x)
        x = layer_norm(x + ff_out)
        return x


class Decoder:
    def __init__(self, num_layers, d_model, num_heads, d_ff, vocab_size, max_len=5000):
        self.embedding = init_weights((vocab_size, d_model))
        self.pos_enc = PositionalEncoding(d_model, max_len)
        self.layers = [DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)]

    def __call__(self, token_ids, enc_output, src_mask=None):
        seq_len = token_ids.shape[1]
        x = self.embedding[token_ids] + self.pos_enc(seq_len)

        tgt_mask = causal_mask(seq_len)

        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        return x

# Quick test
decoder = Decoder(num_layers=2, d_model=16, num_heads=4, d_ff=64, vocab_size=100)
tgt_ids = np.array([[2, 7, 4]])  # batch=1, seq=3
dec_out = decoder(tgt_ids, enc_out)
print("Decoder output shape:", dec_out.shape)
print("\nCausal mask (seq_len=4):")
print(causal_mask(4).astype(int))

Decoder output shape: (1, 3, 16)

Causal mask (seq_len=4):
[[0 1 1 1]
 [0 0 1 1]
 [0 0 0 1]
 [0 0 0 0]]


## 7. Full Transformer

Combines Encoder + Decoder + final linear projection to vocabulary logits.

```
Source tokens → Encoder → encoder_output
                               ↓
Target tokens → Decoder → decoder_output → Linear → logits (vocab_size)
```

In [8]:
class Transformer:
    """
    Full encoder-decoder Transformer.

    Parameters:
        src_vocab_size: source vocabulary size
        tgt_vocab_size: target vocabulary size
        d_model: embedding / hidden dimension
        num_heads: number of attention heads
        d_ff: feed-forward inner dimension
        num_layers: number of encoder and decoder layers
        max_len: maximum sequence length for positional encoding
    """
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=64,
                 num_heads=4, d_ff=256, num_layers=2, max_len=5000):
        self.encoder = Encoder(num_layers, d_model, num_heads, d_ff,
                               src_vocab_size, max_len)
        self.decoder = Decoder(num_layers, d_model, num_heads, d_ff,
                               tgt_vocab_size, max_len)
        self.output_proj = init_weights((d_model, tgt_vocab_size))

    def __call__(self, src_ids, tgt_ids, src_mask=None):
        enc_output = self.encoder(src_ids, src_mask)
        dec_output = self.decoder(tgt_ids, enc_output, src_mask)
        logits = dec_output @ self.output_proj  # (batch, tgt_len, tgt_vocab)
        return logits

    def predict_next(self, logits):
        """Greedy: pick the highest-probability token at each position."""
        probs = softmax(logits, axis=-1)
        return np.argmax(probs, axis=-1)

print("Transformer class ready.")

Transformer class ready.


## 8. Demo — English → French Translation (Forward Pass)

Let's simulate a **machine translation** task to make the data flow concrete.

**Example:** Translate `"the cat sits on the mat"` → `"le chat est assis sur le tapis"`

We build a small vocabulary, convert words to token IDs, pass them through the Transformer,
and see what comes out. The model has **random weights** (not trained), so the output won't
be correct French — but the shapes and data flow are real.

In [9]:
# ══════════════════════════════════════════════════════════════
# STEP 1: Build vocabularies (word → integer ID)
# ══════════════════════════════════════════════════════════════

# English (source) vocabulary
src_vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2,
             "the": 3, "cat": 4, "sits": 5, "on": 6, "mat": 7,
             "dog": 8, "runs": 9, "in": 10, "park": 11}

# French (target) vocabulary
tgt_vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2,
             "le": 3, "chat": 4, "est": 5, "assis": 6, "sur": 7,
             "tapis": 8, "chien": 9, "court": 10, "dans": 11, "parc": 12}

# Reverse lookup: ID → word
src_id2word = {v: k for k, v in src_vocab.items()}
tgt_id2word = {v: k for k, v in tgt_vocab.items()}

print("English vocabulary:", src_vocab)
print("French vocabulary: ", tgt_vocab)

English vocabulary: {'<pad>': 0, '<sos>': 1, '<eos>': 2, 'the': 3, 'cat': 4, 'sits': 5, 'on': 6, 'mat': 7, 'dog': 8, 'runs': 9, 'in': 10, 'park': 11}
French vocabulary:  {'<pad>': 0, '<sos>': 1, '<eos>': 2, 'le': 3, 'chat': 4, 'est': 5, 'assis': 6, 'sur': 7, 'tapis': 8, 'chien': 9, 'court': 10, 'dans': 11, 'parc': 12}


In [13]:
# ══════════════════════════════════════════════════════════════
# STEP 2: Convert sentences to token IDs
# ══════════════════════════════════════════════════════════════

# Input (English):  "the cat sits on the mat"
# Target (French):  "<sos> le chat est assis sur le tapis"
#   (decoder input is shifted right — starts with <sos>)

src_sentence = "the cat sits on the mat"
tgt_sentence = "le chat est assis sur le tapis"

src_tokens = [src_vocab[w] for w in src_sentence.split()]
tgt_tokens = [tgt_vocab["<sos>"]] + [tgt_vocab[w] for w in tgt_sentence.split()]

src_ids = np.array([src_tokens])   # shape: (1, 6)
tgt_ids = np.array([tgt_tokens])   # shape: (1, 8) — includes <sos>

print(f"English input:  {src_sentence}")
print(f"  Token IDs:    {src_tokens}")
print(f"  Meaning:      {[src_id2word[i] for i in src_tokens]}")
print()
print(f"French target (decoder input):  <sos> {tgt_sentence}")
print(f"  Token IDs:    {tgt_tokens}")
print(f"  Meaning:      {[tgt_id2word[i] for i in tgt_tokens]}")

English input:  the cat sits on the mat
  Token IDs:    [3, 4, 5, 6, 3, 7]
  Meaning:      ['the', 'cat', 'sits', 'on', 'the', 'mat']

French target (decoder input):  <sos> le chat est assis sur le tapis
  Token IDs:    [1, 3, 4, 5, 6, 7, 3, 8]
  Meaning:      ['<sos>', 'le', 'chat', 'est', 'assis', 'sur', 'le', 'tapis']


In [14]:
# ══════════════════════════════════════════════════════════════
# STEP 3: Build the Transformer and run forward pass
# ══════════════════════════════════════════════════════════════

model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=64,
    num_heads=4,
    d_ff=256,
    num_layers=2,
)

# Forward pass: encoder reads English, decoder reads French (shifted)
logits = model(src_ids, tgt_ids)
predicted_ids = model.predict_next(logits)[0]

print("=" * 65)
print("TRANSFORMER FORWARD PASS")
print("=" * 65)
print()
print(f"  Encoder input (English): {src_sentence}")
print(f"    IDs: {src_ids[0].tolist()}")
print()
print(f"  Decoder input (French shifted): <sos> {tgt_sentence}")
print(f"    IDs: {tgt_ids[0].tolist()}")
print()
print(f"  Output logits shape: {logits.shape}")
print(f"    → (batch=1, target_len={logits.shape[1]}, french_vocab={logits.shape[2]})")
print()
print("-" * 65)
print("  At each decoder position, the model predicts the NEXT French word:")
print("-" * 65)
decoder_input_words = [tgt_id2word[i] for i in tgt_tokens]
for pos in range(len(tgt_tokens)):
    pred_word = tgt_id2word.get(predicted_ids[pos], f"[{predicted_ids[pos]}]")
    probs = softmax(logits[0, pos])
    confidence = probs[predicted_ids[pos]]
    # Expected next word (what a trained model should predict)
    if pos < len(tgt_sentence.split()):
        expected = tgt_sentence.split()[pos]
    else:
        expected = "<eos>"
    print(f"  Position {pos}: decoder sees {str(decoder_input_words[:pos+1]):40s} → "
          f"predicts '{pred_word}' (expected '{expected}')")

print()
print("  Predicted French: ", " ".join(tgt_id2word.get(i, f"[{i}]") for i in predicted_ids))
print("  Expected French:  ", tgt_sentence, "<eos>")
print()
print("  NOTE: Output is random because the model is NOT trained.")
print("  With training, the predictions would match the expected French.")

TRANSFORMER FORWARD PASS

  Encoder input (English): the cat sits on the mat
    IDs: [3, 4, 5, 6, 3, 7]

  Decoder input (French shifted): <sos> le chat est assis sur le tapis
    IDs: [1, 3, 4, 5, 6, 7, 3, 8]

  Output logits shape: (1, 8, 13)
    → (batch=1, target_len=8, french_vocab=13)

-----------------------------------------------------------------
  At each decoder position, the model predicts the NEXT French word:
-----------------------------------------------------------------
  Position 0: decoder sees ['<sos>']                                → predicts 'court' (expected 'le')
  Position 1: decoder sees ['<sos>', 'le']                          → predicts 'court' (expected 'chat')
  Position 2: decoder sees ['<sos>', 'le', 'chat']                  → predicts 'court' (expected 'est')
  Position 3: decoder sees ['<sos>', 'le', 'chat', 'est']           → predicts 'court' (expected 'assis')
  Position 4: decoder sees ['<sos>', 'le', 'chat', 'est', 'assis']  → predicts 'dans' (

In [15]:
# ══════════════════════════════════════════════════════════════
# STEP 4: Visualize attention & data flow at each stage
# ══════════════════════════════════════════════════════════════

print("=" * 65)
print("DATA FLOW THROUGH THE TRANSFORMER")
print("=" * 65)

# Re-run encoder to show intermediate shapes
enc_output = model.encoder(src_ids)
print(f"""
ENCODER:
  Input words:           {src_sentence.split()}
  Input IDs:             {src_ids[0].tolist()}
  After embedding:       shape {model.encoder.embedding[src_ids].shape}
                         (each word → {model.encoder.embedding.shape[1]}-dim vector)
  + Positional encoding: adds position info to each word
  After 2 encoder layers: shape {enc_output.shape}
                         (6 words × 64-dim contextual representations)
""")

# Re-run decoder
dec_output = model.decoder(tgt_ids, enc_output)
print(f"""DECODER:
  Input words:           {[tgt_id2word[i] for i in tgt_tokens]}
  Input IDs:             {tgt_ids[0].tolist()}
  Causal mask:           Each position can only see itself and earlier tokens
  Cross-attention:       Decoder attends to all 6 encoder positions
  After 2 decoder layers: shape {dec_output.shape}
                         (8 positions × 64-dim hidden states)

OUTPUT PROJECTION:
  Linear layer:          {dec_output.shape} × {model.output_proj.shape} → {logits.shape}
                         Maps 64-dim → {len(tgt_vocab)} French vocabulary words
  Softmax:               Converts logits to probabilities
  Argmax:                Picks highest-probability word at each position
""")

DATA FLOW THROUGH THE TRANSFORMER

ENCODER:
  Input words:           ['the', 'cat', 'sits', 'on', 'the', 'mat']
  Input IDs:             [3, 4, 5, 6, 3, 7]
  After embedding:       shape (1, 6, 64)
                         (each word → 64-dim vector)
  + Positional encoding: adds position info to each word
  After 2 encoder layers: shape (1, 6, 64)
                         (6 words × 64-dim contextual representations)

DECODER:
  Input words:           ['<sos>', 'le', 'chat', 'est', 'assis', 'sur', 'le', 'tapis']
  Input IDs:             [1, 3, 4, 5, 6, 7, 3, 8]
  Causal mask:           Each position can only see itself and earlier tokens
  Cross-attention:       Decoder attends to all 6 encoder positions
  After 2 decoder layers: shape (1, 8, 64)
                         (8 positions × 64-dim hidden states)

OUTPUT PROJECTION:
  Linear layer:          (1, 8, 64) × (64, 13) → (1, 8, 13)
                         Maps 64-dim → 13 French vocabulary words
  Softmax:               Converts 

## 9. Architecture Summary

### Example walkthrough: "the cat sits on the mat" → "le chat est assis sur le tapis"

| Stage | Input | Output | Shape |
|---|---|---|---|
| **Tokenize source** | `"the cat sits on the mat"` | `[3, 4, 5, 6, 3, 7]` | (1, 6) |
| **Tokenize target** | `"<sos> le chat est assis sur le tapis"` | `[1, 3, 4, 5, 6, 7, 3, 8]` | (1, 8) |
| **Embed + PosEnc** | Token IDs | Dense vectors | (1, 6, 64) |
| **Encoder** | Word vectors | Contextual representations | (1, 6, 64) |
| **Decoder** | Target vectors + encoder output | Hidden states | (1, 8, 64) |
| **Output projection** | Hidden states | Logits over French vocab | (1, 8, 13) |
| **Softmax + Argmax** | Logits | Predicted French word IDs | (1, 8) |

### Components

| Component | Parameters | Purpose |
|---|---|---|
| **Positional Encoding** | 0 (fixed sinusoidal) | Injects position information |
| **Multi-Head Attention** | 4 × (d_model × d_model) = 4 × d² | Learns what to attend to |
| **Feed-Forward Network** | d_model × d_ff + d_ff × d_model | Non-linear transformation per position |
| **Encoder Layer** | 1 MHA + 1 FFN | Builds contextual representations |
| **Decoder Layer** | 2 MHA + 1 FFN | Masked self-attn + cross-attn + FFN |
| **Output Projection** | d_model × tgt_vocab | Maps hidden states to vocabulary logits |

**This implementation uses random weights** (no training). In practice, the model would be
trained with cross-entropy loss and an optimizer like Adam with warmup scheduling.